# Synthetic data for Wiki Eval Dataset

This notebook builds a contextual faithfulness dataset from **WikiEval**.

Pipeline:
1. Load WikiEval validation set from Hugging Face.
The orginal datasets contains:
    - `question`: a question that can be answered from the given Wikipedia page (source).
    - `source`: The source Wikipedia page from which the question and context are generated.
    - `grounded_answer`: answer grounded on context_v1
    - `ungrounded_answer`: answer generated without context_v1
    - `poor_answer`: answer with poor relevancy compared to grounded_answer and ungrounded_answer
    - `context_v1`: Ideal context to answer the given question
    - `context_v2`: context that contains redundant information compared to context_v1

2. Parse the dataset into `id`, `question`, `context`, `gold_answer`, `bad_answer`:
   - `id`: unique identifier for each question-answer pair.
   - `question`: the question being asked.
   - `context`: the relevant passage from Wikipedia that contains the information needed to answer the question, use `context_v1`.
   - `gold_answer`: the correct answer extracted from the context, use `grounded_answer`.
   - `bad_answer`: a plausible but incorrect answer generated by a language model, designed to test the model's ability to distinguish between correct and incorrect information, use `poor_answer`.

3. Duplicate the dataset by swapping `gold_answer` and `bad_answer` to create negative examples, ensuring that the negative answer is not supported by the provided context.

3. Export: `labeled_wikieval.csv` with columns: `id`, `question`, `context`, `answer`, `label` (1 for gold_answer, 0 for bad_answer).


In [1]:
import os
import time
import json
import re
from typing import Any, Dict, List, Optional

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset


c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

In [2]:
# ==============================
# Dataset config
# ==============================

DATASET_NAME = "vibrantlabsai/WikiEval"


ORIGINAL_PATH = "../data/wikiEval/wikieval.csv"
LABELED_PATH = "../data/wikiEval/labeled_wikieval.csv"
DEV_PATH = "../data/wikiEval/dev_wikieval.csv"

In [3]:
## 2. Load WikiEval dataset from Hugging Face

In [4]:
ds = load_dataset("vibrantlabsai/WikiEval")
ds = ds["train"]
print(ds)
print(ds.column_names)

Dataset({
    features: ['answer', 'question', 'context_v1', 'context_v2', 'ungrounded_answer', 'source', 'poor_answer'],
    num_rows: 50
})
['answer', 'question', 'context_v1', 'context_v2', 'ungrounded_answer', 'source', 'poor_answer']


In [5]:
ds[0]

{'answer': 'Answer: The PSLV-C56 mission is scheduled to be launched on Sunday, 30 July 2023 at 06:30 IST / 01:00 UTC. It will be launched from the Satish Dhawan Space Centre, Sriharikota, Andhra Pradesh, India.',
 'question': 'Question: When is the scheduled launch date and time for the PSLV-C56 mission, and where will it be launched from?',
 'context_v1': ["The PSLV-C56 is the 58th mission of Indian Space Research Organisation's Polar Satellite Launch Vehicle (PSLV) and the 17th flight of the PSLV-CA variant, and will be get launched from Satish Dhawan Space Centre First Launch Pad ( FLP ).\n\nLaunch\nIt is Scheduled to get launched on Sunday, 30 July 2023 at 06:30 IST / 01:00 UTC from Satish Dhawan Space Centre, Sriharikota, Andhra Pradesh, India. This is a dedicated commercial mission through NSIL with DS-SAR as primary satellite and VELOX-AM as a co-passenger satellite With other 5 Satellites, All satellites from this mission belongs to Singapore."],
 'context_v2': ["The PSLV-C56 

## 3. Convert WikiEval dataset into contextual faithfulness dataset

In [6]:
def _parse_context(context: List[str]) -> str:
    return " ".join(ctx.strip() for ctx in context)

In [7]:
def convert_wikieval_to_faithfulness_dataset(ds) -> pd.DataFrame:
    records = []
    for i, item in tqdm(enumerate(ds), total=len(ds)):
        question = item["question"].replace("Question: ", "").strip()
        context = _parse_context(item["context_v1"])
        gold_answer = item["answer"].replace("Answer: ", "").strip()
        bad_answer = item["poor_answer"]

        # Positive example
        records.append({
            "id": f"{i}_pos",
            "question": question,
            "context": context,
            "answer": gold_answer,
            "label": 1
        })

        # Negative example
        records.append({
            "id": f"{i}_neg",
            "question": question,
            "context": context,
            "answer": bad_answer,
            "label": 0
        })

    df = pd.DataFrame(records)
    return df


In [8]:
labeled_df = convert_wikieval_to_faithfulness_dataset(ds)
print("Labeled WikiEval dataset:")
print(labeled_df.shape)
display(labeled_df.head(3))

100%|██████████| 50/50 [00:00<00:00, 8171.25it/s]

Labeled WikiEval dataset:
(100, 5)


,id,question,context,answer,label
0,0_pos,When is the scheduled launch date and time for...,The PSLV-C56 is the 58th mission of Indian Spa...,The PSLV-C56 mission is scheduled to be launch...,1
1,0_neg,When is the scheduled launch date and time for...,The PSLV-C56 is the 58th mission of Indian Spa...,The scheduled launch date and time for the PSL...,0
2,1_pos,What is the objective of the Uzbekistan-Afghan...,The Uzbekistan–Afghanistan–Pakistan Railway Pr...,The objective of the Uzbekistan-Afghanistan-Pa...,1


Create a Dev set from the WikiEval. The dev set will include:
- `id`: unique identifier for each question-answer pair.
- `question`: the question being asked.
- `context`: the relevant passage from Wikipedia that contains the information needed to answer the question, use `context_v1`.
- `gold_answer`: the correct answer extracted from the context, use `grounded_answer`.
- `predicted_answer`: a plausible but may incorrect answer generated by a language model, designed to test the model's ability to distinguish between correct and incorrect information, use `ungrounded_answer`.

In [12]:
def create_dev(ds) -> pd.DataFrame:
    records = []
    for i, item in tqdm(enumerate(ds), total=len(ds)):
        question = item["question"].replace("Question: ", "").strip()
        context = _parse_context(item["context_v1"])
        gold_answer = item["answer"].replace("Answer: ", "").strip()
        predicted_answer = item["ungrounded_answer"].replace("Answer: ", "").strip()

        records.append({
            "id": f"{i}",
            "question": question,
            "context": context,
            "gold_answer": gold_answer,
            "predicted_answer": predicted_answer
        })

    df = pd.DataFrame(records)
    return df
    

## 4. Finalize and save the dataset

In [13]:
if not os.path.exists("../data/wikiEval"):
    os.makedirs("../data/wikiEval")

In [14]:
raw_df = pd.DataFrame(ds)
raw_df.to_csv(ORIGINAL_PATH, index=False)
labeled_df.to_csv(LABELED_PATH, index=False)
dev_df = create_dev(ds)
dev_df.to_csv(DEV_PATH, index=False)

print(f"Saved original WikiEval dataset to {ORIGINAL_PATH}")
print(raw_df.shape)
display(raw_df.head(3))

print(f"Saved labeled WikiEval dataset to {LABELED_PATH}")
print(labeled_df.shape)
display(labeled_df.head(3)) 

print(f"Saved dev set to {DEV_PATH}")
print(dev_df.shape)
display(dev_df.head(3)) 

100%|██████████| 50/50 [00:00<00:00, 16140.63it/s]

Saved original WikiEval dataset to ../data/wikiEval/wikieval.csv
(50, 7)


,answer,question,context_v1,context_v2,ungrounded_answer,source,poor_answer
0,Answer: The PSLV-C56 mission is scheduled to b...,Question: When is the scheduled launch date an...,[The PSLV-C56 is the 58th mission of Indian Sp...,[The PSLV-C56 is the 58th mission of Indian Sp...,The PSLV-C56 mission is scheduled to be launch...,PSLV-C56,The scheduled launch date and time for the PSL...
1,Answer: The objective of the Uzbekistan-Afghan...,Question: What is the objective of the Uzbekis...,[The Uzbekistan–Afghanistan–Pakistan Railway P...,[The Uzbekistan–Afghanistan–Pakistan Railway P...,The objective of the Uzbekistan-Afghanistan-Pa...,Uzbekistan–Afghanistan–Pakistan Railway Project,The objective of the Uzbekistan-Afghanistan-Pa...
2,Answer: PharmaCann was founded in 2014 by Theo...,Question: When was PharmaCann founded and what...,"[Found in 2014 by Theodore Scott, PharmaCann i...","[Found in 2014 by Theodore Scott, PharmaCann i...",PharmaCann was founded in 2010 by Theodore Wol...,PharmaCann,PharmaCann was founded in 2014.PharmaCann is a...


Saved labeled WikiEval dataset to ../data/wikiEval/labeled_wikieval.csv
(100, 5)


,id,question,context,answer,label
0,0_pos,When is the scheduled launch date and time for...,The PSLV-C56 is the 58th mission of Indian Spa...,The PSLV-C56 mission is scheduled to be launch...,1
1,0_neg,When is the scheduled launch date and time for...,The PSLV-C56 is the 58th mission of Indian Spa...,The scheduled launch date and time for the PSL...,0
2,1_pos,What is the objective of the Uzbekistan-Afghan...,The Uzbekistan–Afghanistan–Pakistan Railway Pr...,The objective of the Uzbekistan-Afghanistan-Pa...,1


Saved dev set to ../data/wikiEval/dev_wikieval.csv
(50, 5)


,id,question,context,gold_answer,predicted_answer
0,0,When is the scheduled launch date and time for...,The PSLV-C56 is the 58th mission of Indian Spa...,The PSLV-C56 mission is scheduled to be launch...,The PSLV-C56 mission is scheduled to be launch...
1,1,What is the objective of the Uzbekistan-Afghan...,The Uzbekistan–Afghanistan–Pakistan Railway Pr...,The objective of the Uzbekistan-Afghanistan-Pa...,The objective of the Uzbekistan-Afghanistan-Pa...
2,2,When was PharmaCann founded and what is its he...,"Found in 2014 by Theodore Scott, PharmaCann is...",PharmaCann was founded in 2014 by Theodore Sco...,PharmaCann was founded in 2010 by Theodore Wol...
